# 03 — 18 Hybrid Retrieval Experiments + LLM

In [ ]:
from pathlib import Path
import pandas as pd
from src.config import load_config, qdrant_client_from_config, llm_models_from_env
from src.qdrant_io import combos
from src.retrieve import retrieve_combo, rrf
from src.llm import generate_answer

cfg = load_config()
client = qdrant_client_from_config(cfg)
llm_models = llm_models_from_env()
questions = pd.read_parquet(cfg["paths"]["questions"])

configs = [
    {"config_id": f"HYB-{i:02d}", "dense_model": d, "sparse_model": s, "late_model": l}
    for i, (d, s, l) in enumerate(combos(cfg), 1)
]
print("Configurations:", len(configs))
print("LLMs:", llm_models)

In [ ]:
rows = []
for q in questions.to_dict("records"):
    for config in configs:
        dense_hits, sparse_hits, late_hits = retrieve_combo(
            client,
            config["dense_model"],
            config["sparse_model"],
            config["late_model"],
            q["question"],
            cfg["rrf"]["top_k"],
        )
        fused = rrf([dense_hits, sparse_hits, late_hits], k=cfg["rrf"]["k"], top_k=cfg["rrf"]["top_k"])

        for llm in llm_models:
            answer, usage = generate_answer(llm, q["question"], fused)
            rows.append({
                "question_id": q.get("question_id"),
                "question": q["question"],
                "config_id": config["config_id"],
                "dense_model": config["dense_model"],
                "sparse_model": config["sparse_model"],
                "late_model": config["late_model"],
                "llm_model": llm,
                "answer": answer,
                "expected_output": q.get("expected_output"),
                "retrieval_context": [x["text"] for x in fused],
                "retrieved_hits": fused,
                "usage": usage,
            })

responses = pd.DataFrame(rows)
print("Responses:", len(responses))

In [ ]:
Path(cfg["paths"]["responses_parquet"]).parent.mkdir(parents=True, exist_ok=True)
responses.to_parquet(cfg["paths"]["responses_parquet"], index=False)
responses.to_json(cfg["paths"]["responses_jsonl"], orient="records", lines=True)